# Goldilocks pilot 01 — hidden-state collection
Collects baseline, perturbed and re-anchored generations with hidden-state summaries and full provenance. Use a GPU runtime.

In [ ]:
!pip -q install transformers accelerate sentencepiece pandas


In [ ]:
import json, time, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # upgrade after pilot
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = Path('/content/goldilocks_runs.jsonl')
SEEDS = [11, 23, 37, 41, 53]
MAX_NEW_TOKENS = 180

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None
)
model.eval()


In [ ]:
TASKS = [
  {
    'id': 'logic_001',
    'task': 'A farmer has 17 sheep. All but 9 run away. How many remain? Explain briefly.',
    'purpose': 'Solve the stated problem exactly. Preserve the quantities and do not invent assumptions.',
    'reference': '9',
    'relevant_analogy': 'Treat all but 9 as the complement: every sheep except 9 leaves.',
    'alternative_cause': 'Check whether the wording describes subtraction or a remaining set.',
    'irrelevant': 'The history of sheep domestication spans thousands of years.'
  },
  {
    'id': 'causal_001',
    'task': 'A service became slower immediately after a deployment. Give the three strongest testable explanations and the first measurement for each.',
    'purpose': 'Produce testable causal hypotheses. Separate observations from assumptions.',
    'reference': '',
    'relevant_analogy': 'Use differential diagnosis: rank causes by temporal fit, mechanism and discriminating measurement.',
    'alternative_cause': 'The deployment may be correlated with, but not responsible for, the slowdown.',
    'irrelevant': 'Some distributed systems are named after animals.'
  }
]

CONDITIONS = [
  ('none', 0, None),
  ('relevant_analogy', 1, 'relevant_analogy'),
  ('relevant_analogy', 2, 'relevant_analogy'),
  ('alternative_cause', 1, 'alternative_cause'),
  ('alternative_cause', 2, 'alternative_cause'),
  ('irrelevant', 1, 'irrelevant'),
  ('irrelevant', 2, 'irrelevant'),
]

def prompt_for(task, kind, strength, phase):
    anchor = f"PURPOSE ANCHOR:\n{task['purpose']}\n\nTASK:\n{task['task']}"
    if phase == 'baseline' or kind == 'none':
        return anchor
    perturbation = task[kind]
    if strength == 2:
        perturbation += ' Explore this perspective thoroughly before deciding.'
    if phase == 'perturbed':
        return anchor + f"\n\nEXPLORATION PULSE:\n{perturbation}"
    return anchor + f"\n\nA prior exploration used this temporary perspective:\n{perturbation}\n\nRe-anchor now. Keep only insights that remain valid under the purpose anchor."


In [ ]:
def last_token_layer_vectors(outputs):
    vectors = []
    for layer in outputs.hidden_states:
        vectors.append(layer[0, -1].detach().float().cpu().numpy())
    return np.stack(vectors)

@torch.inference_mode()
def run_once(prompt, seed):
    set_seed(seed)
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    generated = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=0.7, top_p=0.9,
        return_dict_in_generate=True, output_hidden_states=False
    )
    full = generated.sequences
    answer_ids = full[:, inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(answer_ids[0], skip_special_tokens=True)
    forward = model(full, output_hidden_states=True, use_cache=False)
    layers = last_token_layer_vectors(forward)
    return answer, layers

def digest(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

records = []
for task in TASKS:
    for kind, strength, _ in CONDITIONS:
        for seed in SEEDS:
            phases = ['baseline'] if kind == 'none' else ['baseline', 'perturbed', 'return']
            for phase in phases:
                prompt = prompt_for(task, kind, strength, phase)
                answer, layers = run_once(prompt, seed)
                record = {
                    'run_id': digest(f"{task['id']}|{kind}|{strength}|{seed}|{phase}")[:20],
                    'timestamp_ns': time.time_ns(),
                    'model_id': MODEL_ID,
                    'task_id': task['id'],
                    'condition': kind,
                    'strength': strength,
                    'seed': seed,
                    'phase': phase,
                    'purpose': task['purpose'],
                    'prompt_hash': digest(prompt),
                    'answer': answer,
                    'answer_hash': digest(answer),
                    'layer_vectors': layers.tolist(),
                }
                records.append(record)
                with OUT.open('a', encoding='utf-8') as f:
                    f.write(json.dumps(record, ensure_ascii=False) + '\n')
                print(task['id'], kind, strength, seed, phase)

print(f'Saved {len(records)} records to {OUT}')


In [ ]:
from google.colab import files
files.download(str(OUT))
